# Bakery: A Minimal Cypher Simulation

This notebook builds the same small bakery simulation as `bakery.py`, but in the style most users will probably use while exploring Cypher: one concept at a time in a notebook.

The model is intentionally tiny. It creates one commodity, one source, one sink, one institution, and one region, then exports and runs the simulation through Cyclus.

## Imports

Cypher's handwritten objects live in `cypher`. Discovered Cyclus archetype libraries are imported as modules, such as `cypher.agents` and `cypher.cycamore`. If these imports fail, run `cypher discover` in the active container or Cyclus environment.

In [ ]:
from pathlib import Path

import cypher.agents as agents
import cypher.cycamore as cycamore

import cypher

## Simulation Control

The control block sets the top-level Cyclus simulation settings. This example runs for ten time steps starting in January 2000.

In [ ]:
simulation = cypher.Simulation(
    cypher.Control(
        duration=10,
        start_year=2000,
        start_month=1,
    ),
    name="bakery",
)
simulation.add_library("agents")
simulation.add_library("cycamore")

## Commodity And Recipe

Cyclus facilities exchange named commodities. Recipes describe material compositions. Here the composition is deliberately simple because the example is about the authoring workflow, not fuel-cycle realism.

In [ ]:
toast = cypher.Commodity("Toast")

toast_recipe = cypher.Recipe(
    "Toast",
    basis="atom",
    composition={10030000: 1.0},
)

## Facility Prototypes

The bakery is a Cycamore `Source` that produces the toast commodity. The bread store is a Cycamore `Sink` that consumes it. Cypher accepts the `Commodity` object and writes the commodity name when it exports XML.

In [ ]:
bakery = cycamore.Source(
    "Bakery",
    outcommod=toast,
    throughput=8334,
)

store = cycamore.Sink(
    "Bread Store",
    in_commods=[toast],
    capacity=1000,
)

## Institution And Region

Cyclus simulations place facilities inside institutions and institutions inside regions. This example uses the no-op `NullInst` and `NullRegion` archetypes.

In [ ]:
institution = agents.NullInst("OneInst")
institution.add_initial_facility(bakery)
institution.add_initial_facility(store)

region = agents.NullRegion("OneRegion")
region.add(institution)

## Assemble And Validate

Add the root objects to the simulation, then validate. Cypher follows object references from the region and institution to collect the facilities and archetype declarations.

In [ ]:
simulation.add(toast_recipe, region)
simulation.validate()

## Inspect The XML

`to_xml()` returns the generated hierarchical Cyclus input as a string. In the container, the XML includes the full schema header discovered from `cyclus -n`.

In [ ]:
xml_text = simulation.to_xml()
print(xml_text)

## Run Cyclus

`run()` validates the simulation, writes fresh XML, launches Cyclus, and returns a `RunResult`. The `overwrite=True` flag is included so rerunning this notebook replaces the previous bakery files.

In [ ]:
run_directory = Path("runs") / "bakery-notebook"
result = simulation.run(directory=run_directory, overwrite=True)
result

## Output Paths

Cypher stops at the Cyclus SQLite output. Use Cymetric or other analysis code for post-processing.

In [ ]:
result.input_path, result.output_path